Jerry Zhou (zz977)
---
Kaggle Competition (Word + Character TF-IDF + Linear SVM)

**1. Import necessary libraries**

In [1]:
import pandas as pd
import numpy as np
import re
import time
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import roc_auc_score
from scipy.special import expit
from scipy.sparse import hstack
from itertools import product

**2. Set the hyperparameters**

In [2]:
seed = 42
test_split = 0.2

word_ngram_ranges = [(1, 2), (1, 3), (1, 4), (2, 4)]
max_features = 50000
min_freq = 3
max_freq = 1.0
word_lowercase = False

char_analyzer = "char"
char_ngram_ranges = [(2, 5), (3, 6), (4, 7)]
char_max_features = 100000
char_min_freq = 5
char_lowercase = False

word_weights = [0.5, 0.75, 1.0, 1.25]

reg_values = [0.001, 0.01, 0.03, 0.05, 0.1, 0.3, 0.5, 1.0, 3.0, 5.0]

**3. Load and split the data**

In [3]:
train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")

X = train["text"].fillna("")
y = train["label"]
test_text = test["text"].fillna("")

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=test_split, random_state=seed, stratify=y)

**4. Convert the text to word and character level TF-IDF features**

In [4]:
def build_vectorizers(word_ngrams, char_ngrams):
    word_vec = TfidfVectorizer(analyzer="word",
                               ngram_range=word_ngrams,
                               max_features=max_features,
                               min_df=min_freq,
                               max_df=max_freq,
                               sublinear_tf=True,
                               lowercase=word_lowercase,
                               token_pattern=r"(?u)\b\w+\b|[^\w\s]")

    char_vec = TfidfVectorizer(analyzer=char_analyzer,
                               ngram_range=char_ngrams,
                               max_features=char_max_features,
                               min_df=char_min_freq,
                               sublinear_tf=True,
                               lowercase=char_lowercase)

    return word_vec, char_vec

**5. Train the SVM and test for the best hyperparameter**

In [5]:
best_auc = 0
best_params = None
results = []

search_start_time = time.perf_counter()

for word_ngrams, char_ngrams, weight in product(word_ngram_ranges, char_ngram_ranges, word_weights):

    print(f"Testing word n-grams={word_ngrams}, char n-grams={char_ngrams}, word weight={weight}")

    word_vec, char_vec = build_vectorizers(word_ngrams, char_ngrams)

    Xw_train = word_vec.fit_transform(X_train)
    Xc_train = char_vec.fit_transform(X_train)
    Xw_val = word_vec.transform(X_val)
    Xc_val = char_vec.transform(X_val)

    X_train_tfidf = hstack([weight * Xw_train, Xc_train]).tocsr()
    X_val_tfidf = hstack([weight * Xw_val, Xc_val]).tocsr()

    for c in reg_values:
        model = LinearSVC(C=c, class_weight="balanced",loss="hinge", max_iter=10000)
        training_start_time = time.perf_counter()
        model.fit(X_train_tfidf, y_train)
        training_time = time.perf_counter() - training_start_time

        train_auc = roc_auc_score(y_train, model.decision_function(X_train_tfidf))
        val_auc = roc_auc_score(y_val, model.decision_function(X_val_tfidf))

        results.append({
            "word_ngrams": word_ngrams,
            "char_ngrams": char_ngrams,
            "word_weight": weight,
            "C": c,
            "train_auc": train_auc,
            "val_auc": val_auc,
            "training_time_seconds": training_time
        })

        print(f"  C={c}: Train AUROC {train_auc:.4f} | Validation AUROC {val_auc:.6f} | Training Time: {training_time:.2f} seconds")

        if val_auc > best_auc:
            best_auc = val_auc
            best_params = {
                "word_ngrams": word_ngrams,
                "char_ngrams": char_ngrams,
                "word_weight": weight,
                "C": c
            }

search_time = time.perf_counter() - search_start_time

print(f"Total hyperparameter search time: {search_time:.2f} seconds")

results_df = pd.DataFrame(results).sort_values(by="val_auc", ascending=False).reset_index(drop=True)

print("\nBest hyperparameters:")
print(best_params)
print("Best validation AUROC:", round(best_auc, 6))

print("\nTop 10 combinations:")
display(results_df.head(10))

Testing word n-grams=(1, 2), char n-grams=(2, 5), word weight=0.5
  C=0.001: Train AUROC 0.9266 | Validation AUROC 0.900267 | Training Time: 0.17 seconds
  C=0.01: Train AUROC 0.9266 | Validation AUROC 0.900267 | Training Time: 0.16 seconds
  C=0.03: Train AUROC 0.9454 | Validation AUROC 0.921921 | Training Time: 0.23 seconds
  C=0.05: Train AUROC 0.9639 | Validation AUROC 0.942735 | Training Time: 0.25 seconds
  C=0.1: Train AUROC 0.9802 | Validation AUROC 0.958300 | Training Time: 0.31 seconds
  C=0.3: Train AUROC 0.9961 | Validation AUROC 0.974805 | Training Time: 0.50 seconds
  C=0.5: Train AUROC 0.9988 | Validation AUROC 0.977401 | Training Time: 0.87 seconds
  C=1.0: Train AUROC 0.9999 | Validation AUROC 0.977921 | Training Time: 1.32 seconds
  C=3.0: Train AUROC 1.0000 | Validation AUROC 0.976430 | Training Time: 1.45 seconds
  C=5.0: Train AUROC 1.0000 | Validation AUROC 0.976426 | Training Time: 1.53 seconds
Testing word n-grams=(1, 2), char n-grams=(2, 5), word weight=0.75
  

,word_ngrams,char_ngrams,word_weight,C,train_auc,val_auc,training_time_seconds
0,"(2, 4)","(2, 5)",1.00,0.5,0.999924,0.982173,0.848711
1,"(2, 4)","(2, 5)",1.25,0.5,0.999989,0.981786,0.786232
2,"(2, 4)","(2, 5)",1.00,1.0,1.000000,0.981411,1.068696
3,"(2, 4)","(2, 5)",1.25,0.3,0.999594,0.981273,0.586557
4,"(2, 4)","(2, 5)",0.75,1.0,0.999997,0.981233,1.070035
5,"(1, 4)","(2, 5)",1.00,0.5,0.999920,0.981231,1.013142
6,"(2, 4)","(2, 5)",1.00,5.0,1.000000,0.981142,1.100816
7,"(2, 4)","(2, 5)",1.00,3.0,1.000000,0.981142,1.154833
8,"(1, 3)","(2, 5)",1.00,0.5,0.999923,0.981073,0.704292
9,"(2, 4)","(2, 5)",0.75,0.5,0.999673,0.981071,0.820409


**6. Train final model** 

In [6]:
final_pipeline_start_time = time.perf_counter()

final_word_vec, final_char_vec = build_vectorizers(best_params["word_ngrams"], best_params["char_ngrams"])

X_full_tfidf = hstack([best_params["word_weight"] * final_word_vec.fit_transform(X),final_char_vec.fit_transform(X)]).tocsr()
X_test_tfidf = hstack([best_params["word_weight"] * final_word_vec.transform(test_text),final_char_vec.transform(test_text)]).tocsr()

final_model = LinearSVC(C=best_params["C"], class_weight="balanced",loss="hinge", max_iter=10000)

final_fit_start_time = time.perf_counter()
final_model.fit(X_full_tfidf, y)
final_fit_time = time.perf_counter() - final_fit_start_time

final_pipeline_time = time.perf_counter() - final_pipeline_start_time

print(f"Final SVM training time: {final_fit_time:.2f} seconds")
print(f"Final training pipeline time (TF-IDF + SVM): {final_pipeline_time:.2f} seconds")

Final SVM training time: 1.46 seconds
Final training pipeline time (TF-IDF + SVM): 38.13 seconds


**7. Make predictions and save results**

In [7]:
test_scores = final_model.decision_function(X_test_tfidf)

test_predictions = expit(test_scores)

submission = pd.DataFrame({
    "id": test["id"],
    "label": np.round(test_predictions, 2)
})

submission.to_csv("tfidf_svm_submission.csv", index=False)
print(submission.shape)
print(submission.head())

(3000, 2)
      id  label
0  13001   0.78
1  13002   0.28
2  13003   0.41
3  13004   0.25
4  13005   0.76
